# Model Evaluation
We begin by re-fitting the optimal Support Vector Classifier (SVC) model using the results of `RandomizedSearchCV`. Additional topics covered: Model Statistics, Model Testing, Model Interpretation, and Feature Importance in Individual Predictions.

### 1. Load Libraries & Setup

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.inspection import permutation_importance

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
%matplotlib inline

### 2. Re-fit Optimal Model from RandomizedSearchCV Results

In [6]:
# Best parameters extracted from RandomizedSearchCV
best_params = {
    'C': 10.0,
    'gamma': 'scale',
    'kernel': 'rbf',
    'probability': True,
    'random_state': 42
}

# Re-fit optimal model on full training set
optimal_svc = SVC(**best_params)
optimal_svc.fit(X_train, y_train)

print('Optimal SVC Model successfully re-fitted.')

NameError: name 'X_train' is not defined

### 3. Model Statistics & Metrics

In [ ]:
# Predictions on training data
y_train_pred = optimal_svc.predict(X_train)
y_train_prob = optimal_svc.predict_proba(X_train)[:, 1]

# Calculate statistics
stats_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC'],
    'Training Score': [
        accuracy_score(y_train, y_train_pred),
        precision_score(y_train, y_train_pred),
        recall_score(y_train, y_train_pred),
        f1_score(y_train, y_train_pred),
        roc_auc_score(y_train, y_train_prob)
    ]
})

print('--- Model Training Statistics ---')
print(stats_df.to_string(index=False))

print('\nDetailed Training Classification Report:')
print(classification_report(y_train, y_train_pred))

### 4. Model Testing (Out-of-Sample Evaluation)

In [ ]:
# Predictions on test set
y_test_pred = optimal_svc.predict(X_test)
y_test_prob = optimal_svc.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title('Test Set Confusion Matrix', fontsize=12)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.show()

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
test_auc = roc_auc_score(y_test, y_test_prob)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Test ROC Curve (AUC = {test_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Out-of-Sample ROC Curve')
plt.legend(loc='lower right')
plt.show()

### 5. Model Interpretation (Global Permutation Importance)

In [3]:
perm_importance = permutation_importance(optimal_svc, X_test, y_test, n_repeats=10, random_state=42)
sorted_idx = perm_importance.importances_mean.argsort()

plt.figure(figsize=(8, 5))
plt.barh(range(len(sorted_idx)), perm_importance.importances_mean[sorted_idx], align='center')
plt.yticks(range(len(sorted_idx)), [X_test.columns[i] for i in sorted_idx])
plt.xlabel('Permutation Importance (Drop in Mean Accuracy)')
plt.title('Global Model Interpretation: Permutation Feature Importance')
plt.show()

NameError: name 'X_test' is not defined

### 6. Feature Importance in Individual Predictions

In [4]:
def explain_individual_prediction(model, sample_df, feature_names):
    base_score = model.decision_function(sample_df)[0]
    predicted_class = model.predict(sample_df)[0]
    predicted_prob = model.predict_proba(sample_df)[0][predicted_class]

    local_impacts = {}
    for col in feature_names:
        perturbed_sample = sample_df.copy()
        perturbed_sample[col] = 0.0
        perturbed_score = model.decision_function(perturbed_sample)[0]
        local_impacts[col] = base_score - perturbed_score

    impact_df = pd.DataFrame(list(local_impacts.items()), columns=['Feature', 'Local Impact'])
    impact_df = impact_df.sort_values(by='Local Impact', key=abs, ascending=False)

    print(f'Predicted Class: {predicted_class} (Probability: {predicted_prob:.4f})')
    print(f'Base Decision Function Score: {base_score:.4f}')

    plt.figure(figsize=(8, 4))
    sns.barplot(x='Local Impact', y='Feature', data=impact_df, palette='vlag')
    plt.title('Local Feature Contributions for Individual Prediction')
    plt.xlabel('Marginal Decision Impact (Score Shift)')
    plt.show()
    return impact_df

# Explain sample instance #0
explain_individual_prediction(optimal_svc, X_test.iloc[[0]], X_test.columns)

NameError: name 'X_test' is not defined